#  Finding Paths for Scene 61 and 103

In [ ]:
from nuscenes.nuscenes import NuScenes
import os

# Load the nuScenes mini dataset
nusc = NuScenes(version='v1.0-mini', dataroot='/Users/junseok/Downloads/v1.0-mini', verbose=False)

# Target scene names
target_scenes = ['scene-0061', 'scene-0103']

# List to store the result paths
selected_sample_data_paths = []

# Iterate through all scenes
for scene in nusc.scene:
    if scene['name'] in target_scenes:
        print(f"Processing {scene['name']}: {scene['description']}")
        sample_token = scene['first_sample_token']

        # Traverse all samples in the scene
        while sample_token:
            sample = nusc.get('sample', sample_token)
            lidar_token = sample['data']['LIDAR_TOP']  # Get the LIDAR_TOP token
            sample_data = nusc.get('sample_data', lidar_token)
            file_path = sample_data['filename']       # Extract the file path
            selected_sample_data_paths.append(file_path)

            # Move to the next sample in the scene
            sample_token = sample['next'] if sample['next'] != '' else None

# Print the collected file paths (e.g., samples/LIDAR_TOP/xxx.pcd.bin)
for path in selected_sample_data_paths:
    print(path)


Processing scene-0061: Parked truck, construction, intersection, turn left, following a van
Processing scene-0103: Many peds right, wait for turning car, long bike rack left, cyclist
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402927647951.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402928147847.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402928698048.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402929197353.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402929697797.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402930152601.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402930648325.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402931198511.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402931697833.pcd.bin
samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LID

# Randomly extract a single frame path from the scenes

In [ ]:
import numpy as np
from nuscenes.nuscenes import NuScenes
import os

# Load the nuScenes mini dataset
nusc = NuScenes(version='v1.0-mini', dataroot='/Users/junseok/Downloads/v1.0-mini', verbose=False)

# Target scenes to search
target_scenes = ['scene-0061', 'scene-0103']

# Dictionary to store the selected frame for each scene
frame_indices = {}

# Iterate through all scenes in the dataset
for scene in nusc.scene:
    if scene['name'] in target_scenes:
        sample_token = scene['first_sample_token']
        frame_tokens = []

        # Traverse all samples in the scene and collect frame file paths
        while sample_token:
            sample = nusc.get('sample', sample_token)
            lidar_token = sample['data']['LIDAR_TOP']
            sample_data = nusc.get('sample_data', lidar_token)
            frame_tokens.append(sample_data['filename'])
            sample_token = sample['next'] if sample['next'] else None

        # Select the middle frame (or roughly middle)
        middle_frame = frame_tokens[len(frame_tokens)//2]
        frame_indices[scene['name']] = middle_frame

# Print the selected frame paths
print("🎯 Selected frames:")
for scene_name, frame_path in frame_indices.items():
    print(f"{scene_name} → {frame_path}")


🎯 선택된 프레임:
scene-0061 → samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin
scene-0103 → samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin


# Single Frame Visualization

In [ ]:
import numpy as np
import open3d as o3d
import os

def load_bin_as_pcd(bin_path):
    # nuScenes .pcd.bin files have float32 values: x, y, z, intensity (4 or 5 floats per point)
    points = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 5)[:, :3]  # Use only x, y, z
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # Assign color (optional)
    pcd.paint_uniform_color([0, 0, 0])  # Black color (change if needed)

    return pcd

# Set base path to dataset
base_path = "/Users/junseok/Downloads/v1.0-mini"

# Specify the target frame files for scene-0061 and scene-0103
scene_0061_file = os.path.join(base_path, "samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin")
scene_0103_file = os.path.join(base_path, "samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin")

# Load point clouds from each file
pcd_0061 = load_bin_as_pcd(scene_0061_file)
pcd_0103 = load_bin_as_pcd(scene_0103_file)

# Visualize each frame
print("🚗 Visualizing scene-0061 frame...")
o3d.visualization.draw_geometries([pcd_0061], window_name="Scene 0061 Frame", width=800, height=600)

print("🚲 Visualizing scene-0103 frame...")
o3d.visualization.draw_geometries([pcd_0103], window_name="Scene 0103 Frame", width=800, height=600)


/opt/miniconda3/envs/python3_10/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🚗 Visualizing scene-0061 frame...
🚲 Visualizing scene-0103 frame...


In [ ]:
import numpy as np
import open3d as o3d
import os

def load_bin_as_colored_pcd(bin_path):
    # nuScenes .pcd.bin file contains float32 values in the format: x, y, z, intensity (total 5 floats)
    points = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 5)
    xyz = points[:, :3]
    z_vals = xyz[:, 2]  # Use z-axis (height) for coloring

    # Color mapping: normalize z values and map to a colormap (mimicking jet/viridis colormap)
    z_min, z_max = np.min(z_vals), np.max(z_vals)
    z_norm = (z_vals - z_min) / (z_max - z_min + 1e-6)  # Normalize to [0, 1]
    colors = plt.get_cmap('viridis')(z_norm)[:, :3]     # Get RGB from colormap

    # Create colored point cloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.colors = o3d.utility.Vector3dVector(colors)

    return pcd

# Import matplotlib for colormap usage
import matplotlib.pyplot as plt

# Set the dataset base path
base_path = "/Users/junseok/Downloads/v1.0-mini"

# Define the file paths for specific frames in scene-0061 and scene-0103
scene_0061_file = os.path.join(base_path, "samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin")
scene_0103_file = os.path.join(base_path, "samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin")

# Load and convert the binary point cloud files into colored Open3D point clouds
pcd_0061 = load_bin_as_colored_pcd(scene_0061_file)
pcd_0103 = load_bin_as_colored_pcd(scene_0103_file)

# Visualize the colored point clouds
print("🚗 Visualizing scene-0061 frame with color...")
o3d.visualization.draw_geometries([pcd_0061], window_name="Scene 0061 Frame (Color)", width=800, height=600)

print("🚲 Visualizing scene-0103 frame with color...")
o3d.visualization.draw_geometries([pcd_0103], window_name="Scene 0103 Frame (Color)", width=800, height=600)


/opt/miniconda3/envs/python3_10/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🚗 Visualizing scene-0061 frame with color...
🚲 Visualizing scene-0103 frame with color...


## ✅ Selected Frame Summary: Representative Cases for Preprocessing, Sampling, and Inference Experiments

The two selected frames are highly suitable as representative cases for preprocessing, sampling, and inference experiments due to their contrasting characteristics.

---

### 🔹 scene-0061 Frame
- **Total Objects**: 156 (dense scene)
- **Classes Present**:  
  Vehicles, Pedestrians, Trucks, Bicycles, Motorcycles, Traffic Cones, Barriers, etc.
- **Environment**:  
  Complex urban setting with high object diversity and a wide range of object scales.

---

### 🔹 scene-0103 Frame
- **Total Objects**: 61 (relatively sparse scene)
- **Classes Present**:  
  Mostly Vehicles, some Pedestrians, Pushable Objects, and Traffic Cones.
- **Environment**:  
  Likely a highway or major road scenario — more open and less cluttered than scene-0061.

---


## Check Which Objects Are Present in Each Frame

In [ ]:
from nuscenes.nuscenes import NuScenes
import os

# Initialize nuScenes with the mini dataset
nusc = NuScenes(version='v1.0-mini', dataroot='/Users/junseok/Downloads/v1.0-mini', verbose=False)

# List of selected LIDAR_TOP file names
target_filenames = [
    'samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin',
    'samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin'
]

# Loop through each target file
for filename in target_filenames:
    print(f"\n🔍 Checking annotations for: {filename}")

    # Find the corresponding sample_data object
    sample_data = next(sd for sd in nusc.sample_data if sd['filename'] == filename)
    sample_token = sample_data['sample_token']

    # Retrieve the sample object associated with this data
    sample = nusc.get('sample', sample_token)

    # Get annotation tokens for all objects in this sample
    ann_tokens = sample['anns']
    print(f"Total {len(ann_tokens)} objects found:")

    # Print annotation details for each object
    for ann_token in ann_tokens:
        ann = nusc.get('sample_annotation', ann_token)
        name = ann['category_name']              # Object class name
        translation = ann['translation']         # Object position (x, y, z)
        size = ann['size']                       # Object size (width, length, height)
        yaw = ann['rotation']                    # Object rotation (quaternion)
        print(f"🟢 Class: {name} | Location: {translation[:2]} | Size: {size} | Rotation: {yaw}")



🔍 Checking annotations for: samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin
총 156개 객체 존재:
🟢 Class: human.pedestrian.adult | 위치: [373.208, 1130.504] | 크기: [0.621, 0.669, 1.642] | 회전: [0.9831098797903927, 0.0, 0.0, -0.18301629506281616]
🟢 Class: human.pedestrian.adult | 위치: [374.221, 1143.758] | 크기: [0.775, 0.769, 1.711] | 회전: [-0.5008537539567612, 0.0, 0.0, 0.8655319272836908]
🟢 Class: vehicle.car | 위치: [354.091, 1132.468] | 크기: [2.011, 4.633, 1.573] | 회전: [0.9814385473495932, 0.0, 0.0, -0.19177689583555196]
🟢 Class: human.pedestrian.adult | 위치: [380.559, 1170.193] | 크기: [0.752, 0.819, 1.637] | 회전: [0.7616909960501779, 0.0, 0.0, 0.6479404498378597]
🟢 Class: vehicle.bicycle | 위치: [372.914, 1129.598] | 크기: [0.689, 1.77, 1.709] | 회전: [0.994910649043305, 0.0, 0.0, -0.10076110569177861]
🟢 Class: human.pedestrian.adult | 위치: [373.356, 1141.411] | 크기: [0.661, 0.703, 1.839] | 회전: [-0.4053011480242389, 0.0, 0.0, 0.9141832307640706]
🟢 Class: human.pedestrian.

## Save the Two Frames Separately (Organize File Paths)

In [ ]:
import os
import shutil

# Root path of the nuScenes dataset
nusc_root = '/Users/junseok/Downloads/v1.0-mini'

# Destination folder to save selected frames
save_dir = '/Users/junseok/Downloads/selected_frames'
os.makedirs(save_dir, exist_ok=True)

# Relative paths of the selected LIDAR_TOP frame files
selected_files = [
    'samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin',
    'samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin'
]

# Copy each selected file to the save directory
for rel_path in selected_files:
    src_path = os.path.join(nusc_root, rel_path)                   # Source path in dataset
    dst_path = os.path.join(save_dir, os.path.basename(rel_path)) # Destination path (filename only)
    shutil.copy(src_path, dst_path)                               # Copy the file
    print(f"✅ Copied to {dst_path}")                              # Confirmation message


✅ Copied to /Users/junseok/Downloads/selected_frames/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin
✅ Copied to /Users/junseok/Downloads/selected_frames/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin


##  Code to Convert .pcd.bin Point Cloud Files to .npy Format

In [ ]:
import numpy as np
import os

# Source folder containing the .pcd.bin files to convert
src_dir = '/Users/junseok/Downloads/selected_frames'

# Destination folder to save the converted .npy files
dst_dir = '/Users/junseok/Downloads/selected_npy'
os.makedirs(dst_dir, exist_ok=True)

# List of .pcd.bin files to convert
bin_files = [
    'n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin',
    'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.pcd.bin'
]

# Perform conversion for each file
for fname in bin_files:
    bin_path = os.path.join(src_dir, fname)                           # Full path to source .bin file
    npy_path = os.path.join(dst_dir, fname.replace('.pcd.bin', '.npy'))  # Destination .npy path

    # nuScenes .pcd.bin format: float32 values representing (x, y, z, intensity, ring_index)
    pointcloud = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 5)  # Load as (N, 5) array
    np.save(npy_path, pointcloud)                                        # Save as .npy file

    print(f"✅ Saved: {npy_path} | shape: {pointcloud.shape}")           # Print confirmation and shape


FileNotFoundError: [Errno 2] No such file or directory: '/Users/junseok/Downloads/selected_frames/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.pcd.bin'

In [ ]:
import numpy as np
import open3d as o3d

# 🔹 Set the path to the .npy point cloud file for visualization
npy_path = '/Users/junseok/Downloads/selected_npy/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.npy'

# 🔹 Load the .npy file and extract only x, y, z coordinates
points = np.load(npy_path)[:, :3]  # Extract x, y, z only (ignore intensity and other values)

# 🔹 Create an Open3D PointCloud object
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# 🔹 Launch the Open3D visualizer
o3d.visualization.draw_geometries([pcd])


# Downsampling Operation

In [ ]:
import os
import numpy as np
import open3d as o3d

# Source directory containing original .npy files
src_dir = '/Users/junseok/Downloads/selected_npy'

# Destination root directory for downsampled .npy files
dst_root = '/Users/junseok/Downloads/downsampled_npy'
os.makedirs(dst_root, exist_ok=True)

# Target .npy files to process
npy_files = [
    'n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.npy',
    'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.npy'
]

# Downsampling ratios to apply
ratios = [1.0, 0.5, 0.25]

# Corresponding voxel sizes for each ratio
voxel_sizes = {
    1.0: 0.05,   # Dense, close to original
    0.5: 0.2,
    0.25: 0.4,
}

def random_sampling(pc, ratio):
    """
    Randomly samples a portion of the point cloud.
    :param pc: (N, D) NumPy array of original point cloud
    :param ratio: float between 0 and 1
    :return: downsampled point cloud of shape (int(N * ratio), D)
    """
    n = int(len(pc) * ratio)
    indices = np.random.choice(len(pc), size=n, replace=False)
    return pc[indices]

def voxel_downsampling_with_preserved_dims(pc, voxel_size):
    """
    Perform voxel-based downsampling using Open3D,
    while preserving additional channels like intensity or ring index by averaging.
    
    :param pc: (N, 5) or (N, 4), with additional features in columns 4 and 5
    :param voxel_size: voxel size to use for downsampling
    :return: (M, 5) downsampled point cloud with features preserved
    """
    from collections import defaultdict

    xyz = pc[:, :3]       # Extract x, y, z
    others = pc[:, 3:]    # Extract additional features (e.g., intensity, ring_index)

    # Open3D voxel downsampling on spatial coordinates
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    down = pcd.voxel_down_sample(voxel_size=voxel_size)
    down_xyz = np.asarray(down.points)

    # Compute voxel indices for each point
    voxel_indices = np.floor((xyz - xyz.min(0)) / voxel_size).astype(np.int32)
    voxel_dict = defaultdict(list)
    for i, voxel in enumerate(voxel_indices):
        voxel_dict[tuple(voxel)].append(i)

    # For each downsampled point, average the corresponding feature values
    down_others = []
    for point in down_xyz:
        voxel = tuple(np.floor((point - xyz.min(0)) / voxel_size).astype(np.int32))
        indices = voxel_dict[voxel]
        avg = others[indices].mean(axis=0)
        down_others.append(avg)

    down_others = np.stack(down_others)
    return np.hstack([down_xyz, down_others])  # Combine xyz and averaged features


# Main processing loop
for fname in npy_files:
    npy_path = os.path.join(src_dir, fname)
    pointcloud = np.load(npy_path)

    for ratio in ratios:
        # Random Sampling
        rs_pc = random_sampling(pointcloud, ratio)
        rs_dir = os.path.join(dst_root, f'RS_{int(ratio * 100)}')  # e.g., RS_50
        os.makedirs(rs_dir, exist_ok=True)
        np.save(os.path.join(rs_dir, fname), rs_pc)
        print(f"✅ RS-{int(ratio * 100)}% saved: {rs_pc.shape}")

        # Voxel-based Downsampling
        voxel_pc = voxel_downsampling_with_preserved_dims(pointcloud, voxel_sizes[ratio])
        voxel_dir = os.path.join(dst_root, f'Voxel_{int(ratio * 100)}')  # e.g., Voxel_25
        os.makedirs(voxel_dir, exist_ok=True)
        np.save(os.path.join(voxel_dir, fname), voxel_pc)
        print(f"✅ Voxel-{int(ratio * 100)}% saved: {voxel_pc.shape}")


✅ RS-100% saved: (34688, 5)


/var/folders/tz/ybx0mhh15t5gprqcrlcs03380000gn/T/ipykernel_57783/278925652.py:55: RuntimeWarning: Mean of empty slice.
  avg = others[indices].mean(axis=0)


✅ Voxel-100% saved: (22282, 5)
✅ RS-50% saved: (17344, 5)
✅ Voxel-50% saved: (11675, 5)
✅ RS-25% saved: (8672, 5)
✅ Voxel-25% saved: (7630, 5)
✅ RS-100% saved: (34752, 5)
✅ Voxel-100% saved: (24160, 5)
✅ RS-50% saved: (17376, 5)
✅ Voxel-50% saved: (12522, 5)
✅ RS-25% saved: (8688, 5)
✅ Voxel-25% saved: (7228, 5)


# Upsampling Operation

In [ ]:
import numpy as np

def upsample_gaussian_jittering(points: np.ndarray, target_num_points: int, std=0.01) -> np.ndarray:
    """
    Upsample a point cloud by duplicating points and adding Gaussian noise to reach the target number of points.
    
    :param points: (N, 5) point cloud (x, y, z, intensity, ring index)
    :param target_num_points: Desired number of points after upsampling (e.g., 100% of original)
    :param std: Standard deviation of the Gaussian noise added to x, y, z
    :return: (target_num_points, 5) upsampled point cloud
    """
    current_num = len(points)
    assert target_num_points > current_num, "Target must be greater than current number of points"

    # Number of points to duplicate
    dup_num = target_num_points - current_num
    dup_indices = np.random.choice(current_num, size=dup_num, replace=True)
    dup_points = points[dup_indices].copy()

    # Add Gaussian noise to x, y, z only
    noise = np.random.normal(scale=std, size=(dup_num, 3))
    dup_points[:, :3] += noise

    # Combine original and duplicated points
    upsampled = np.vstack([points, dup_points])
    return upsampled


from scipy.spatial import cKDTree

def voxel_based_interpolation(points: np.ndarray, target_num_points: int, voxel_size: float = 0.5) -> np.ndarray:
    """
    Perform voxel-based interpolation to generate additional points and reach the target number.
    
    :param points: (N, 4) or (N, 5) point cloud (x, y, z, intensity [, ring index])
    :param target_num_points: Desired number of points after interpolation
    :param voxel_size: Voxel grid size used for grouping points
    :return: (target_num_points, 5) interpolated and padded/truncated point cloud
    """
    xyz = points[:, :3]

    # Normalize feature dimensions: ensure intensity + dummy (if only 4 columns)
    if points.shape[1] == 4:
        intensities = np.hstack([points[:, 3:4], np.zeros((points.shape[0], 1))])  # Add dummy channel
    elif points.shape[1] == 5:
        intensities = points[:, 3:5]
    else:
        raise ValueError("points must be of shape (N, 4) or (N, 5)")

    # Step 1: Convert to voxel coordinates
    min_bound = xyz.min(axis=0)
    voxel_indices = np.floor((xyz - min_bound) / voxel_size).astype(np.int32)

    # Step 2: For each voxel, compute one interpolated point using average of first two points
    voxel_dict = {}
    for idx, voxel in enumerate(voxel_indices):
        key = tuple(voxel)
        voxel_dict.setdefault(key, []).append(idx)

    interpolated_points = []
    for idx_list in voxel_dict.values():
        if len(idx_list) < 2:
            continue  # Need at least 2 points to interpolate
        pts = xyz[idx_list]
        mid = (pts[0] + pts[1]) / 2  # Midpoint of first two points
        int_avg = intensities[idx_list[:2]].mean(axis=0)  # Average features
        interpolated_points.append(np.concatenate([mid, int_avg]))

    # Step 3: Combine original and interpolated points
    interpolated_points = np.array(interpolated_points)
    if interpolated_points.size == 0:
        total = np.hstack([xyz, intensities])
    else:
        total = np.vstack([points, interpolated_points])

    # Step 4: Adjust total size to match target
    if len(total) > target_num_points:
        return total[np.random.choice(len(total), size=target_num_points, replace=False)]
    elif len(total) < target_num_points:
        pad_idx = np.random.choice(len(total), size=target_num_points - len(total), replace=True)
        return np.vstack([total, total[pad_idx]])
    else:
        return total


In [ ]:
import numpy as np
import os

# Assumes upsampling functions are already implemented elsewhere
# def upsample_gaussian_jittering(...)
# def voxel_based_interpolation(...)

# Target number of points to upsample to (based on full resolution of scene-0061)
target_100 = 34688

# Base directory for downsampled .npy files
base_dir = '/Users/junseok/Downloads/downsampled_npy'

# Directory to save upsampled results
save_root = '/Users/junseok/Downloads/upsampled_npy'

# Dictionary of scenes and their corresponding filenames
scenes = {
    "scene-0061": "n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.npy",
    "scene-0103": "n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.npy"
}

# Helper function to save upsampled results
def save_upsampled(points, method_name, filename):
    save_dir = os.path.join(save_root, method_name)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, filename)
    np.save(save_path, points)
    print(f"✅ Saved: {save_path} | shape: {points.shape}")

# Process all scenes
for scene_name, fname in scenes.items():
    print(f"\n🔧 Processing {scene_name} ...")

    # Paths to 25% and 50% randomly sampled point clouds
    rs25_path = os.path.join(base_dir, 'RS_25', fname)
    rs50_path = os.path.join(base_dir, 'RS_50', fname)

    # Load the downsampled point clouds
    points_25 = np.load(rs25_path)
    points_50 = np.load(rs50_path)

    # 1. RS25 → 100%, using Gaussian Jittering
    upsampled_25_gj = upsample_gaussian_jittering(points_25, target_100, std=0.02)
    save_upsampled(upsampled_25_gj, 'RS25_to100_GJ', f'{scene_name}_RS25_to100_GJ.npy')

    # 2. RS25 → 100%, using Voxel-Based Interpolation
    upsampled_25_vi = voxel_based_interpolation(points_25, target_100, voxel_size=0.4)
    save_upsampled(upsampled_25_vi, 'RS25_to100_VI', f'{scene_name}_RS25_to100_VI.npy')

    # 3. RS50 → 100%, using Gaussian Jittering
    upsampled_50_gj = upsample_gaussian_jittering(points_50, target_100, std=0.02)
    save_upsampled(upsampled_50_gj, 'RS50_to100_GJ', f'{scene_name}_RS50_to100_GJ.npy')

    # 4. RS50 → 100%, using Voxel-Based Interpolation
    upsampled_50_vi = voxel_based_interpolation(points_50, target_100, voxel_size=0.4)
    save_upsampled(upsampled_50_vi, 'RS50_to100_VI', f'{scene_name}_RS50_to100_VI.npy')

print("\n🎉 All scene upsampling results saved successfully!")



🔧 Processing scene-0061 ...
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS25_to100_GJ/scene-0061_RS25_to100_GJ.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS25_to100_VI/scene-0061_RS25_to100_VI.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS50_to100_GJ/scene-0061_RS50_to100_GJ.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS50_to100_VI/scene-0061_RS50_to100_VI.npy | shape: (34688, 5)

🔧 Processing scene-0103 ...
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS25_to100_GJ/scene-0103_RS25_to100_GJ.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS25_to100_VI/scene-0103_RS25_to100_VI.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS50_to100_GJ/scene-0103_RS50_to100_GJ.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/upsampled_npy/RS50_to100_VI/scene-0103_RS50_to100_VI.npy | shape: (34688, 5)

🎉 모든 scene 업샘플링 결과 저장 완료!


## Upsampled Files: Code to Convert .npy to .bin

In [ ]:
import os
import numpy as np

# Set input and output directories
npy_root = '/Users/junseok/Downloads/upsampled_npy'   # Directory containing .npy files
bin_root = '/Users/junseok/Downloads/upsampled_bin'   # Directory to save converted .bin files
os.makedirs(bin_root, exist_ok=True)

# Subdirectories for different upsampling methods
method_dirs = [
    'RS25_to100_GJ',  # Random Sampling 25% → 100% using Gaussian Jittering
    'RS25_to100_VI',  # Random Sampling 25% → 100% using Voxel Interpolation
    'RS50_to100_GJ',  # Random Sampling 50% → 100% using Gaussian Jittering
    'RS50_to100_VI'   # Random Sampling 50% → 100% using Voxel Interpolation
]

# Function to convert .npy to .bin in nuScenes format
def convert_npy_to_bin(npy_path, bin_path):
    points = np.load(npy_path)
    # Save as float32 binary file (nuScenes format: x, y, z, intensity, ring index)
    points.astype(np.float32).tofile(bin_path)
    print(f"✅ Converted: {npy_path} → {bin_path} | shape: {points.shape}")

# Process all method subfolders
for method in method_dirs:
    npy_dir = os.path.join(npy_root, method)
    bin_dir = os.path.join(bin_root, method)
    os.makedirs(bin_dir, exist_ok=True)

    # Convert all .npy files in the directory
    for fname in os.listdir(npy_dir):
        if fname.endswith('.npy'):
            npy_path = os.path.join(npy_dir, fname)
            bin_name = fname.replace('.npy', '.bin')
            bin_path = os.path.join(bin_dir, bin_name)
            convert_npy_to_bin(npy_path, bin_path)

print("\n🎉 All .npy → .bin conversions completed successfully!")


✅ Converted: /Users/junseok/Downloads/upsampled_npy/RS25_to100_GJ/scene-0103_RS25_to100_GJ.npy → /Users/junseok/Downloads/upsampled_bin/RS25_to100_GJ/scene-0103_RS25_to100_GJ.bin | shape: (34688, 5)
✅ Converted: /Users/junseok/Downloads/upsampled_npy/RS25_to100_GJ/scene-0061_RS25_to100_GJ.npy → /Users/junseok/Downloads/upsampled_bin/RS25_to100_GJ/scene-0061_RS25_to100_GJ.bin | shape: (34688, 5)
✅ Converted: /Users/junseok/Downloads/upsampled_npy/RS25_to100_VI/scene-0103_RS25_to100_VI.npy → /Users/junseok/Downloads/upsampled_bin/RS25_to100_VI/scene-0103_RS25_to100_VI.bin | shape: (34688, 5)
✅ Converted: /Users/junseok/Downloads/upsampled_npy/RS25_to100_VI/scene-0061_RS25_to100_VI.npy → /Users/junseok/Downloads/upsampled_bin/RS25_to100_VI/scene-0061_RS25_to100_VI.bin | shape: (34688, 5)
✅ Converted: /Users/junseok/Downloads/upsampled_npy/RS50_to100_GJ/scene-0103_RS50_to100_GJ.npy → /Users/junseok/Downloads/upsampled_bin/RS50_to100_GJ/scene-0103_RS50_to100_GJ.bin | shape: (34688, 5)
✅ Con

## Downsampled Files: .npy → .bin Conversion

In [ ]:
import os
import numpy as np

# Source directory containing downsampled .npy files
src_root = '/Users/junseok/Downloads/downsampled_npy'

# Destination directory to save converted .bin files
dst_root = '/Users/junseok/Downloads/downsampled_bin'
os.makedirs(dst_root, exist_ok=True)

# Subdirectories to process (Random Sampling and Voxel-based downsampling)
sub_dirs = [
    'RS_100', 'RS_50', 'RS_25',
    'Voxel_100', 'Voxel_50', 'Voxel_25'
]

# Mapping from original filenames to scene identifiers
# Used to rename files with more meaningful scene names
scene_map = {
    'n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.npy': 'scene-0061',
    'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.npy': 'scene-0103',
}

# Loop through each subdirectory
for sub_dir in sub_dirs:
    full_src_dir = os.path.join(src_root, sub_dir)  # Source path
    full_dst_dir = os.path.join(dst_root, sub_dir)  # Destination path
    os.makedirs(full_dst_dir, exist_ok=True)

    # Iterate over all .npy files in the current subdirectory
    for fname in os.listdir(full_src_dir):
        if not fname.endswith('.npy'):
            continue  # Skip non-.npy files

        npy_path = os.path.join(full_src_dir, fname)
        points = np.load(npy_path)

        # Extract scene name and create new filename
        scene_id = scene_map[fname]                      # e.g., scene-0061
        method = sub_dir.replace('_', '')                # e.g., RS25 or Voxel50
        new_fname = f'{scene_id}_{method}.bin'           # e.g., scene-0061_RS25.bin
        bin_path = os.path.join(full_dst_dir, new_fname)

        # Save as float32 binary file
        points.astype(np.float32).tofile(bin_path)
        print(f"✅ Converted: {bin_path} | shape: {points.shape}")


✅ Converted: /Users/junseok/Downloads/downsampled_bin/RS_100/scene-0103_RS100.bin | shape: (34752, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/RS_100/scene-0061_RS100.bin | shape: (34688, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/RS_50/scene-0103_RS50.bin | shape: (17376, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/RS_50/scene-0061_RS50.bin | shape: (17344, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/RS_25/scene-0103_RS25.bin | shape: (8688, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/RS_25/scene-0061_RS25.bin | shape: (8672, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/Voxel_100/scene-0103_Voxel100.bin | shape: (24160, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/Voxel_100/scene-0061_Voxel100.bin | shape: (22282, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/Voxel_50/scene-0103_Voxel50.bin | shape: (12522, 5)
✅ Converted: /Users/junseok/Downloads/downsampled_bin/Voxel_50/scene-0061_Vo

In [ ]:
import os

# Target scenes with corresponding file name base (without extension)
scene_dict = {
    'scene-0061': 'n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682',
    'scene-0103': 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020',
}

# Output path for saving the mapping text file
save_path = '/Users/junseok/Downloads/scene_key.txt'

# Write scene name to filename mapping to a text file
with open(save_path, 'w') as f:
    for scene_name, filename_base in scene_dict.items():
        f.write(f'{scene_name} = {filename_base}\n')

print(f"✅ scene_key.txt saved successfully: {save_path}")


✅ scene_key.txt 저장 완료: /Users/junseok/Downloads/scene_key.txt


# Code to Print the Point Shape of All Files in bin Folders

In [ ]:
import os
import numpy as np

# Root directory containing bin folders
root_dir = "/Users/junseok/Downloads/bin"

# Function to load a .bin file and return its shape (automatically handles 4D or 5D formats)
def load_bin_shape(filepath):
    data = np.fromfile(filepath, dtype=np.float32)
    for dim in [5, 4]:  # Try 5D first, then 4D
        if data.size % dim == 0:
            return data.reshape(-1, dim).shape
    return (data.shape[0], )  # Fallback in case dimensions don't match

# Check and print the shapes of all .bin files in a given folder
def check_bin_shapes_in_folder(folder_path):
    print(f"\n📁 Folder: {folder_path}")
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith(".bin"):
            fpath = os.path.join(folder_path, fname)
            try:
                shape = load_bin_shape(fpath)
                print(f"  - {fname}: shape = {shape}")
            except Exception as e:
                print(f"  - {fname}: ⚠️ Error reading file: {e}")

# Traverse all relevant subdirectories and check .bin file shapes
def walk_all_bin_folders(root_dir):
    for category in ["downsampled_bin", "original_bin", "upsampled_bin"]:
        category_path = os.path.join(root_dir, category)
        if not os.path.exists(category_path):
            continue

        # original_bin contains only .bin files (no subfolders)
        if category == "original_bin":
            check_bin_shapes_in_folder(category_path)
            continue

        # downsampled_bin and upsampled_bin contain subfolders with .bin files
        for subfolder in sorted(os.listdir(category_path)):
            subfolder_path = os.path.join(category_path, subfolder)
            if os.path.isdir(subfolder_path):
                check_bin_shapes_in_folder(subfolder_path)

# Run the bin file shape checker
walk_all_bin_folders(root_dir)



📁 Folder: /Users/junseok/Downloads/bin/downsampled_bin/RS_100
  - scene-0061_RS100.bin: shape = (34688, 5)
  - scene-0103_RS100.bin: shape = (34752, 5)

📁 Folder: /Users/junseok/Downloads/bin/downsampled_bin/RS_25
  - scene-0061_RS25.bin: shape = (8672, 5)
  - scene-0103_RS25.bin: shape = (8688, 5)

📁 Folder: /Users/junseok/Downloads/bin/downsampled_bin/RS_50
  - scene-0061_RS50.bin: shape = (17344, 5)
  - scene-0103_RS50.bin: shape = (17376, 5)

📁 Folder: /Users/junseok/Downloads/bin/downsampled_bin/Voxel_100
  - scene-0061_Voxel100.bin: shape = (22282, 5)
  - scene-0103_Voxel100.bin: shape = (24160, 5)

📁 Folder: /Users/junseok/Downloads/bin/downsampled_bin/Voxel_25
  - scene-0061_Voxel25.bin: shape = (7630, 5)
  - scene-0103_Voxel25.bin: shape = (7228, 5)

📁 Folder: /Users/junseok/Downloads/bin/downsampled_bin/Voxel_50
  - scene-0061_Voxel50.bin: shape = (11675, 5)
  - scene-0103_Voxel50.bin: shape = (12522, 5)

📁 Folder: /Users/junseok/Downloads/bin/original_bin
  - n008-2018-08-0

# Voxel Dimension Adjustment

In [ ]:
def voxel_based_interpolation(points: np.ndarray, target_num_points: int, voxel_size: float = 0.5) -> np.ndarray:
    """
    Generate interpolated points using a voxel grid, then adjust to match the target number of points.
    
    :param points: (N, 4) or (N, 5) point cloud (x, y, z, intensity [, ring index])
    :param target_num_points: Desired number of points after interpolation
    :param voxel_size: Size of the voxel grid used to group nearby points
    :return: (target_num_points, 5) array containing interpolated point cloud with intensity and ring index
    """
    xyz = points[:, :3]  # Extract x, y, z coordinates

    # Normalize the feature dimension to ensure it's always (N, 2): [intensity, ring index or dummy]
    if points.shape[1] == 4:
        # Add dummy channel (zero column) if only intensity is available
        intensities = np.hstack([points[:, 3:4], np.zeros((points.shape[0], 1))])
    elif points.shape[1] == 5:
        intensities = points[:, 3:5]
    else:
        raise ValueError("points must be of shape (N, 4) or (N, 5)")

    # Step 1: Compute voxel indices for each point
    min_bound = xyz.min(axis=0)
    voxel_indices = np.floor((xyz - min_bound) / voxel_size).astype(np.int32)

    # Step 2: Group points by voxel and generate new points via interpolation
    voxel_dict = {}
    for idx, voxel in enumerate(voxel_indices):
        key = tuple(voxel)
        voxel_dict.setdefault(key, []).append(idx)

    interpolated_points = []

    # For each voxel with at least 2 points, interpolate one midpoint
    for idx_list in voxel_dict.values():
        if len(idx_list) < 2:
            continue
        pts = xyz[idx_list]
        mid = (pts[0] + pts[1]) / 2  # Compute midpoint of first two points
        int_avg = intensities[idx_list[:2]].mean(axis=0)  # Average intensity and ring index
        interpolated_points.append(np.concatenate([mid, int_avg]))

    # Convert interpolated list to array
    interpolated_points = np.array(interpolated_points)

    ## Step 3: Merge original and interpolated points
    if interpolated_points.size == 0:
        # If no interpolation occurred, just use original points
        total = np.hstack([points[:, :3], intensities])  # (N, 3) + (N, 2) → (N, 5)
    else:
        # Concatenate original (normalized to 5D if needed) and interpolated points
        total = np.vstack([
            points if points.shape[1] == 5 else np.hstack([points[:, :3], intensities]),
            interpolated_points
        ])

    # Step 4: Adjust total number of points to match the target
    if len(total) > target_num_points:
        # Randomly select a subset if too many
        return total[np.random.choice(len(total), size=target_num_points, replace=False)]
    elif len(total) < target_num_points:
        # Duplicate random points to reach the target
        pad_idx = np.random.choice(len(total), size=target_num_points - len(total), replace=True)
        return np.vstack([total, total[pad_idx]])
    else:
        return total  # Exact match — return as-is


In [ ]:
# ▶️ Set base directories
base_dir = "/Users/junseok/Downloads/npy/downsampled_npy"       # Directory containing downsampled .npy files
save_root = "/Users/junseok/Downloads/npy/upsampled_npy"        # Directory to save upsampled .npy files
target_100 = 34688                                              # Target number of points after upsampling

# Scene name to filename mapping
scenes = {
    "scene-0061": "n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402937198682.npy",
    "scene-0103": "n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151613398020.npy"
}

# ▶️ Function to save the upsampled point cloud to file
def save_upsampled(points, method_name, filename):
    save_dir = os.path.join(save_root, method_name)
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, filename)
    np.save(path, points)
    print(f"✅ Saved: {path} | shape: {points.shape}")

# ▶️ Main execution loop (only for Voxel_25 and Voxel_50 directories)
for scene_name, fname in scenes.items():
    print(f"\n🔧 Processing {scene_name}...")

    # 1. Load Voxel_25 point cloud
    voxel25_path = os.path.join(base_dir, "Voxel_25", fname)
    pts_25 = np.load(voxel25_path)
    # Upsample to target size using voxel-based interpolation
    up_25 = voxel_based_interpolation(pts_25, target_100, voxel_size=0.4)
    # Save the result
    save_upsampled(up_25, "Voxel25_to100_VI", f"{scene_name}_Voxel25_to100_VI.npy")

    # 2. Load Voxel_50 point cloud
    voxel50_path = os.path.join(base_dir, "Voxel_50", fname)
    pts_50 = np.load(voxel50_path)
    # Upsample to target size using voxel-based interpolation
    up_50 = voxel_based_interpolation(pts_50, target_100, voxel_size=0.4)
    # Save the result
    save_upsampled(up_50, "Voxel50_to100_VI", f"{scene_name}_Voxel50_to100_VI.npy")

print("\n🎉 Voxel upsampling completed (Voxel25 and Voxel50 → 100%)")



🔧 Processing scene-0061...
✅ Saved: /Users/junseok/Downloads/npy/upsampled_npy/Voxel25_to100_VI/scene-0061_Voxel25_to100_VI.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/npy/upsampled_npy/Voxel50_to100_VI/scene-0061_Voxel50_to100_VI.npy | shape: (34688, 5)

🔧 Processing scene-0103...
✅ Saved: /Users/junseok/Downloads/npy/upsampled_npy/Voxel25_to100_VI/scene-0103_Voxel25_to100_VI.npy | shape: (34688, 5)
✅ Saved: /Users/junseok/Downloads/npy/upsampled_npy/Voxel50_to100_VI/scene-0103_Voxel50_to100_VI.npy | shape: (34688, 5)

🎉 Voxel 업샘플링 완료 (Voxel25, Voxel50 → 100%)


In [ ]:
import numpy as np
import os

# List of input folders containing .npy files to be converted
npy_folders = [
    "/Users/junseok/Downloads/npy/upsampled_npy/Voxel25_to100_VI",
    "/Users/junseok/Downloads/npy/upsampled_npy/Voxel50_to100_VI"
]

# Root output folder for saving converted .bin files
bin_root = "/Users/junseok/Downloads/bin/upsampled_bin"

# Conversion function: .npy → .bin
def convert_npy_to_bin(npy_dir):
    folder_name = os.path.basename(npy_dir)  # Get last folder name (e.g., "Voxel25_to100_VI")
    bin_dir = os.path.join(bin_root, folder_name)  # Create corresponding output subdirectory
    os.makedirs(bin_dir, exist_ok=True)

    # Iterate through all .npy files in the folder
    for fname in os.listdir(npy_dir):
        if fname.endswith(".npy"):
            npy_path = os.path.join(npy_dir, fname)
            bin_path = os.path.join(bin_dir, fname.replace(".npy", ".bin"))

            points = np.load(npy_path)                         # Load point cloud from .npy
            points.astype(np.float32).tofile(bin_path)         # Save as binary .bin (float32)

            print(f"✅ {fname} → {os.path.basename(bin_path)} | shape: {points.shape}")

# Execute conversion for each folder
for npy_folder in npy_folders:
    print(f"\n🔄 Converting: {npy_folder}")
    convert_npy_to_bin(npy_folder)

print("\n🎉 All conversions complete: .npy → .bin saved successfully!")



🔄 변환 중: /Users/junseok/Downloads/npy/upsampled_npy/Voxel25_to100_VI
✅ scene-0103_Voxel25_to100_VI.npy → scene-0103_Voxel25_to100_VI.bin | shape: (34688, 5)
✅ scene-0061_Voxel25_to100_VI.npy → scene-0061_Voxel25_to100_VI.bin | shape: (34688, 5)

🔄 변환 중: /Users/junseok/Downloads/npy/upsampled_npy/Voxel50_to100_VI
✅ scene-0061_Voxel50_to100_VI.npy → scene-0061_Voxel50_to100_VI.bin | shape: (34688, 5)
✅ scene-0103_Voxel50_to100_VI.npy → scene-0103_Voxel50_to100_VI.bin | shape: (34688, 5)

🎉 모든 변환 완료: .npy → .bin 저장됨!
